# V014: cold grouped E confirmation of the deep curve-loss mixture

Frozen from completed V013 A results and independent audit74/74. Winner: deep0.25 +deep_curve0.25 +compact/curve/memory/offset0.125 each. Reference: the accepted five-member recipe (deep0.5; others0.125). A MAE0.186264237 vs0.190851029.

A new RC-stratified grouped split uses seed20261203 inside original Afit. All six members cold-fit on54,323 breaths;6,791 E-validation breaths and original A-validation remain excluded from these fits. No old A/D models score E. Exact A-selected budgets/rates are frozen. E pressure is loaded only after immutable choice/split locking; scores are computed only after all member fits complete. Pass requires positive MAE gain and wholly negative whole-breath bootstrap interval. No weights/epochs/runner-up search on E.

Default execution is inert. Modeling stays in notebooks. This stage never fits final all-data models or submits. Preserve official0.18025 and all previous files. E is resampling evidence after repeated research on A, not a pristine test.


In [ ]:
import os, json, hashlib, copy, gc, math, fcntl, time
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
V014_PROTOCOL_SHA='dee5840ee16262dc282ebeb98643cb24e312e55761634f8fae074ed6c921f6f1'
V014_V013_SOURCE_SHA='56cb9d7e824e58126bddd71c7c0f7063844974c31ee00a6dac23f2112657615a'
V014_NOTEBOOK='confirmation_v014.ipynb'
V014_OUT='outputs/confirmation_v014'

def v014_require(c,message):
    if not bool(c):raise ValueError(message)
def v014_code(p):
    n=json.loads(Path(p).read_text());return '\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code')
def v014_sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def v014_sig(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def v014_immutable_json(p,v,resume):
    if p.exists():v014_require(resume and json.loads(p.read_text())==v,'Immutable file changed: '+str(p))
    else:v013_json(p,v)
def v014_immutable_array(p,v):
    if p.exists():v014_require(np.array_equal(np.load(p,allow_pickle=False),v),'Existing array differs: '+str(p))
    else:v013_array(p,v)
def v014_memory_factory(spec,n_features,prep):return v8_memory_model(spec,n_features,prep)
def v014_offset_factory(spec,n_features,prep):return v9_offset_model(spec,n_features,prep)
def v014_plain_factory(spec,n_features,prep):return V5PressureRNN(spec,n_features)
for _factory in [v014_memory_factory,v014_offset_factory,v014_plain_factory]:_factory.__module__='confirmation_v014'
def v014_factory(name):
    if name=='memory':return v014_memory_factory
    if name=='offset':return v014_offset_factory
    if name in ['curve','deep_curve']:return v014_plain_factory
    raise ValueError('Unknown V8 member: '+name)

def v014_metrics(p,y):
    e=np.abs(p-y);return dict(mae=float(e.mean()),median_absolute_error=float(np.median(e)),p90_absolute_error=float(np.quantile(e,.9)))
def run_confirmation_v014(root='.',resume=False):
    root=Path(root).resolve();out=root/V014_OUT;out.mkdir(parents=True,exist_ok=True);locks=[];began=time.monotonic()
    try:
        for p in [root/'outputs/.v011_gpu.lock',out/'.run.lock']:
            f=p.open('a+');fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(f)
        v014_require(v014_sha(root/'research/confirmation_protocol_v014.json')==V014_PROTOCOL_SHA,'E protocol changed')
        plan=json.loads((root/'research/confirmation_protocol_v014.json').read_text())
        for p,h in plan['input_hashes'].items():v014_require(v014_sha(root/p)==h,'E prerequisite changed: '+p)
        v014_require(v014_sha(root/'data/train.csv')==plan['train_SHA256'],'Approved data changed')
        v014_require(v014_sha(root/plan['best_CSV'])==plan['best_CSV_SHA256'],'Accepted official CSV changed')
        helper=v014_code(root/'controlled_experiments_v013.ipynb')
        v014_require(hashlib.sha256(helper.encode()).hexdigest()==V014_V013_SOURCE_SHA,'Helper notebook changed')
        v014_require(os.environ.get('PA3_RUN_V013','0')!='1','Disable A-screen run flag')
        exec(compile(helper,'controlled_experiments_v013.ipynb','exec'),globals())
        source=hashlib.sha256(v014_code(root/V014_NOTEBOOK).encode()).hexdigest()
        v014_require(source==os.environ.get('PA3_V014_SOURCE_SHA256'),'E source differs from reviewed pin')
        for filename,h in plan['source_code_hashes'].items():
            code=v014_code(root/'research'/filename);v014_require(hashlib.sha256(code.encode()).hexdigest()==h,'Component changed: '+filename)
            exec(compile(code,filename,'exec'),globals())
        torch.set_num_threads(4);v014_require(torch.cuda.is_available(),'Actual approved CUDA required');seed_sequence(plan['seed'])
        v014_require(torch.are_deterministic_algorithms_enabled(),'Strict determinism required')
        summary=json.loads((root/'outputs/controlled_experiments_v013/screen_summary.json').read_text());review=json.loads((root/'research/screen_review_v013.json').read_text())
        eligible=[c for c in summary['candidates'] if c['gain']>=plan['minimum_A_gain'] and c['interval']['high']<0]
        winner=min(eligible,key=lambda c:(c['mae'],c['name'])) if eligible else None
        v014_require(review['status']=='passed' and winner is not None and winner==summary['selected_for_confirmation']==review['selected'] and winner['weights']==plan['winner_weights'] and winner['name']==plan['winner_name'],'Not the frozen lowest eligible A choice')
        if (out/'confirmation_result.json').exists():
            done=json.loads((out/'confirmation_result.json').read_text())
            previous=json.loads((out/'locked_before_E.json').read_text())
            v014_require(resume and done['status']=='complete' and done['source_code_sha256']==source and done['protocol_sha256']==V014_PROTOCOL_SHA and done['selection_sha256']==v014_sig(previous) and done['locked_file_sha256']==v014_sha(out/'locked_before_E.json') and done['winner_weights']==plan['winner_weights'],'Completed E result changed')
            for record in done['records'].values():
                for path,h in [(record['checkpoint_file'],record['checkpoint_sha256']),(record['report_file'],record['report_sha256'])]:v014_require(v014_sha(root/path)==h,'Completed E artifact changed')
            for artifact in done['prediction_artifacts'].values():v014_require(v014_sha(root/artifact['file'])==artifact['file_sha256'],'Completed E predictions changed')
            return done
        # Read only known inputs before writing the immutable E lock. Pressure
        # labels are loaded later, after choice/split/epochs/weights are frozen.
        known=pd.read_csv(root/'data/train.csv',usecols=['id','breath_id','R','C','time_step','u_in','u_out'],dtype={k:v for k,v in V013_DTYPES.items() if k!='pressure'})
        bids=_validate_feature_frame_v005(known);v014_require(len(bids)==67905,'Approved group count changed')
        with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz',allow_pickle=False) as z:
            afit=np.sort(pd.Index(bids).get_indexer(z['A_fit']));aval=np.sort(pd.Index(bids).get_indexer(z['A_validation']))
        v014_require(len(afit)==61114 and len(aval)==6791 and not np.intersect1d(afit,aval).size and np.array_equal(np.sort(np.r_[afit,aval]),np.arange(67905)),'Original A grouping changed')
        rc=known[['R','C']].to_numpy()[::80];joint=np.searchsorted([5,20,50],rc[:,0])*3+np.searchsorted([10,20,50],rc[:,1])
        from sklearn.model_selection import train_test_split
        fit,val=train_test_split(afit,test_size=6791,random_state=20261203,stratify=joint[afit]);fit=np.sort(fit);val=np.sort(val)
        v014_require(len(fit)==54323 and len(val)==6791 and not np.intersect1d(fit,val).size and not np.intersect1d(np.r_[fit,val],aval).size,'E containment/leakage')
        split=dict(seed=20261203,indices=dict(A_fit=afit.tolist(),A_validation=aval.tolist(),E_fit=fit.tolist(),E_validation=val.tolist()),breath_ids=dict(E_fit=bids[fit].tolist(),E_validation=bids[val].tolist()),group_hashes=dict(E_fit=v013_ah(bids[fit]),E_validation=v013_ah(bids[val])),counts=dict(E_fit=len(fit),E_validation=len(val)))
        v014_immutable_json(out/'E_split.json',split,resume)
        locked=dict(protocol_sha256=V014_PROTOCOL_SHA,source_code_sha256=source,winner_name=plan['winner_name'],winner_weights=plan['winner_weights'],baseline_weights=plan['baseline_weights'],members=plan['members'],input_hashes=plan['input_hashes'],component_hashes=plan['source_code_hashes'],E_split_file_sha256=v014_sha(out/'E_split.json'),train_sha256=plan['train_SHA256'],acceptance=plan['acceptance'],bootstrap_seed=plan['bootstrap_seed'],bootstrap_draws=1000,no_runner_up=True,no_E_epoch_or_weight_selection=True,cold_fit_all_members=True)
        v014_immutable_json(out/'locked_before_E.json',locked,resume);selection=v014_sig(locked)
        sources=dict(plan['source_code_hashes'],**{V014_NOTEBOOK:source,'controlled_experiments_v013.ipynb':V014_V013_SOURCE_SHA,'confirmation_protocol_v014.json':V014_PROTOCOL_SHA,'locked_before_E.json':selection})
        v013_event(out,'E_locked_before_pressure_read',selection_sha256=selection,gpu=torch.cuda.get_device_name(0),pod=os.environ.get('HOSTNAME'))
        labels=pd.read_csv(root/'data/train.csv',usecols=['pressure'],dtype=dict(pressure='float32')).pressure.to_numpy().reshape(-1,80)
        uo=known.u_out.to_numpy(copy=True).reshape(-1,80);ids=known.id.to_numpy(copy=True).reshape(-1,80)
        x,names,feature_bids=make_sequence_features(known,'combined');del known;gc.collect()
        v014_require(x.shape==(67905,80,47) and np.array_equal(feature_bids,bids),'Known feature order/schema changed')
        records={}
        for name,recipe in plan['members'].items():
            stage=out/'E_models'/name;stage.mkdir(parents=True,exist_ok=True);checkpoint=stage/'model.pt';rp=stage/'record.json'
            identity=dict(selection_sha256=selection,member_id=name,recipe=recipe,training_group_sha256=split['group_hashes']['E_fit'],source_fingerprints=sources)
            if rp.exists():
                record=json.loads(rp.read_text());v014_require(resume and record['identity']==identity,'Completed member identity changed')
                for path,h in [(record['checkpoint_file'],record['checkpoint_sha256']),(record['report_file'],record['report_sha256'])]:v014_require(v014_sha(root/path)==h,'Completed member artifact changed')
            else:
                v013_event(out,'cold_E_fit_started',member=name,epochs=recipe['selected_epochs'])
                args=(x[fit],labels[fit],uo[fit],recipe['model_spec'],recipe['train_cfg'],recipe['seed'],recipe['lr_schedule'],checkpoint)
                opts=dict(epoch_log_path=stage/'epochs.jsonl',feature_names=names,device='cuda',resume=resume and _last_path(checkpoint).exists())
                if name in ['deep','original_compact']:report=fit_fixed(*args,**opts)
                else:report=fit_fixed_v008(*args,**opts,model_factory=v014_factory(name),loss_kind=recipe['loss_kind'],arm='v014_E_'+name,source_fingerprints=sources)
                v014_require(report['completed'] and report['executed_epochs']==recipe['selected_epochs'] and report['training_breaths']==54323 and not report['validation_used'] and report['executed_lr_schedule']==recipe['lr_schedule'],'Cold fixed E fit incomplete/changed')
                v013_json(stage/'report.json',report)
                record=dict(identity=identity,identity_sha256=v014_sig(identity),checkpoint_file=str(checkpoint.relative_to(root)),checkpoint_sha256=v014_sha(checkpoint),report_file=str((stage/'report.json').relative_to(root)),report_sha256=v014_sha(stage/'report.json'),training_breaths=54323,selected_epochs=recipe['selected_epochs'],elapsed_seconds=report['elapsed_seconds'])
                v013_json(rp,record)
            records[name]=record;gc.collect();torch.cuda.empty_cache()
        # Confirmation scoring begins only after all six cold fits completed.
        mask=uo[val]==0;truth=labels[val][mask].astype(np.float64);rows=ids[val][mask];counts=mask.sum(1);predictions={};artifacts={}
        v014_immutable_array(out/'E_row_ids.npy',rows)
        for name,recipe in plan['members'].items():
            checkpoint=root/records[name]['checkpoint_file'];stage=checkpoint.parent
            if name in ['deep','original_compact']:p=predict_checkpoint(checkpoint,x[val],device='cuda',batch_size=512)
            else:p=predict_checkpoint_v008(checkpoint,x[val],device='cuda',batch_size=512,model_factory=v014_factory(name),expected_arm='v014_E_'+name,expected_loss_kind=recipe['loss_kind'],source_fingerprints=sources)
            p=p[mask].astype(np.float64);v014_require(p.shape==truth.shape and np.isfinite(p).all(),'Nonfinite/misaligned E predictions')
            pp=stage/'E_predictions.npy';v014_immutable_array(pp,p);predictions[name]=p
            artifacts[name]=dict(file=str(pp.relative_to(root)),file_sha256=v014_sha(pp),array_sha256=v013_ah(p),row_ids_sha256=v013_ah(rows))
        baseline=v013_blend(predictions,plan['baseline_weights'],len(rows));candidate=v013_blend(predictions,plan['winner_weights'],len(rows))
        metrics=dict(baseline=v014_metrics(baseline,truth),candidate=v014_metrics(candidate,truth));interval=v013_interval(np.abs(candidate-truth),np.abs(baseline-truth),counts,plan['bootstrap_seed'])
        gain=metrics['baseline']['mae']-metrics['candidate']['mae'];passed=bool(gain>0 and interval['high']<0)
        v014_immutable_array(out/'baseline_E_predictions.npy',baseline);v014_immutable_array(out/'candidate_E_predictions.npy',candidate)
        v014_require(v014_sha(root/plan['best_CSV'])==plan['best_CSV_SHA256'],'Official best CSV changed')
        for p,h in plan['input_hashes'].items():v014_require(v014_sha(root/p)==h,'Frozen input changed during E: '+p)
        v014_require(hashlib.sha256(v014_code(root/V014_NOTEBOOK).encode()).hexdigest()==source,'E source changed')
        result=dict(status='complete',schema='v014-E-result-v1',selection_sha256=selection,locked_file_sha256=v014_sha(out/'locked_before_E.json'),protocol_sha256=V014_PROTOCOL_SHA,source_code_sha256=source,winner_name=plan['winner_name'],winner_weights=plan['winner_weights'],baseline_weights=plan['baseline_weights'],metrics=metrics,gain=gain,interval=interval,passed=passed,records=records,prediction_artifacts=artifacts,E_scored_points=len(rows),E_truth_array_sha256=v013_ah(truth),E_row_ids_array_sha256=v013_ah(rows),source_fingerprints=sources,cold_fit_all_members=True,no_E_epoch_or_weight_selection=True,original_A_validation_excluded=True,confirmation_is_resampling_evidence=True,no_runner_up_after_failure=True,final_fit_performed=False,submission_performed=False,official_best_preserved=.18025,elapsed_seconds=time.monotonic()-began,completed_utc=datetime.now(timezone.utc).isoformat())
        v014_immutable_json(out/'confirmation_result.json',result,resume=False)
        v013_event(out,'confirmation_complete',passed=passed,baseline_MAE=metrics['baseline']['mae'],candidate_MAE=metrics['candidate']['mae'],gain=gain)
        return result
    finally:
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get('PA3_RUN_V014','0')=='1':
    V014_RESULT=run_confirmation_v014(Path.cwd(),resume=os.environ.get('PA3_RESUME_V014','0')=='1')
    print(json.dumps({k:v for k,v in V014_RESULT.items() if k not in ['records','prediction_artifacts','source_fingerprints']},indent=2))
